# Предсказание одобрения кредита

**Итоговый проект по дисциплине «Анализ данных на Python»**

## Цель исследования

Построить модель машинного обучения, которая по данным заявителя предсказывает, будет ли одобрена заявка на кредит. **Кредитный скоринг (CIBIL score) намеренно исключён** — модель строится исключительно на финансовых характеристиках заявителя и макроэкономическом контексте ЦБ РФ.

## Источники данных

1. **Kaggle — Loan Approval Prediction Dataset** — признаки заявителей и решение банка (Approved / Rejected).
2. **Банк России — SOAP API KeyRateXML** — история ключевой ставки.
3. **Банк России — Data Service API** — средневзвешенные ставки по кредитам физлиц.

> Персональные данные реальных заявителей банки не публикуют. Для обучения модели используется открытый синтетический датасет, обогащённый макроэкономическим контекстом ЦБ РФ.

## 1. Подготовка окружения

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import chi2_contingency, mannwhitneyu

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_collection import build_enriched_dataset
from src.modeling import cross_validate_model, get_best_tree_model_name, train_and_evaluate
from src.preprocessing import clean_loan_data, engineer_features, get_feature_matrix, prepare_target

FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.figsize": (10, 6), "font.size": 11})

print(f"Рабочая директория: {PROJECT_ROOT}")

## 2. Сбор и обогащение данных

Загружаем датасет Kaggle и дополняем его макропоказателями Банка России:
- ключевая ставка (SOAP API);
- средневзвешенная ставка по кредитам физлиц (REST API).

Так как в датасете нет даты подачи заявки, каждой записи назначается синтетический месяц (детерминированно по `loan_id`), после чего подтягиваются макропоказатели за этот период.

CIBIL score удаляется сразу после загрузки — до любого анализа.

In [ ]:
df = build_enriched_dataset()

# Исключаем CIBIL score: цель — оценить предсказуемость одобрения
# без кредитного скоринга, только по финансовым показателям заявителя.
df = df.drop(columns=["cibil_score"], errors="ignore")

print(f"Размер датасета: {df.shape[0]} строк, {df.shape[1]} столбцов")
df.head()

In [ ]:
df.info()
print("\nПропуски:")
print(df.isna().sum()[df.isna().sum() > 0])

**Промежуточный вывод:** датасет содержит 4 269 заявок, 15 столбцов (без CIBIL score). Основные признаки заполнены; 71 пропуск в `key_rate` будет заполнен медианой на этапе препроцессинга.

## 3. Разведочный анализ данных (EDA)

In [ ]:
target_counts = df["loan_status"].str.strip().value_counts()
approval_rate = target_counts.get("Approved", 0) / len(df)
print(target_counts)
print(f"\nДоля одобренных заявок: {approval_rate:.1%}")

In [ ]:
fig, ax = plt.subplots()
sns.barplot(x=target_counts.index, y=target_counts.values, ax=ax,
            hue=target_counts.index, legend=False)
ax.set_title("Распределение решений по заявкам на кредит")
ax.set_xlabel("Статус заявки")
ax.set_ylabel("Количество заявок")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_target_distribution.png", dpi=150)
plt.show()

In [ ]:
numeric_summary = df.select_dtypes(include="number").describe().T
numeric_summary

> ⚠️ **Аномалия данных:** столбец `residential_assets_value` содержит отрицательные значения (минимум −100 000). В реальности стоимость активов не может быть отрицательной — это артефакт синтетического датасета. Такие значения будут обрезаны методом IQR на этапе очистки.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

cols_to_plot = [
    "income_annum", "loan_amount", "loan_term",
    "residential_assets_value", "commercial_assets_value", "bank_asset_value",
]
labels = [
    "Годовой доход", "Сумма кредита", "Срок кредита (лет)",
    "Стоимость недвижимости", "Коммерч. недвижимость", "Банковские активы",
]

plot_df = df.copy()
plot_df["loan_status"] = plot_df["loan_status"].str.strip()

for ax, col, label in zip(axes, cols_to_plot, labels):
    sns.histplot(data=plot_df, x=col, hue="loan_status", ax=ax, bins=30, alpha=0.7)
    ax.set_title(label)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20)

plt.suptitle("Распределение числовых признаков по статусу заявки", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_feature_distributions.png", dpi=150)
plt.show()

### Проверка гипотез

**Гипотеза 1:** заявители с более высоким годовым доходом получают одобрение чаще.

**Гипотеза 2:** образование и статус самозанятости не связаны с решением банка.

In [ ]:
approved_income = df[df["loan_status"].str.strip() == "Approved"]["income_annum"]
rejected_income = df[df["loan_status"].str.strip() == "Rejected"]["income_annum"]

stat, p_value = mannwhitneyu(approved_income, rejected_income, alternative="greater")
print(f"Mann-Whitney U: statistic={stat:.0f}, p-value={p_value:.4f}")
print(f"Медиана дохода (одобрено):  {approved_income.median():>12,.0f} руб.")
print(f"Медиана дохода (отказ):     {rejected_income.median():>12,.0f} руб.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_df = df.copy()
plot_df["loan_status"] = plot_df["loan_status"].str.strip()

sns.violinplot(data=plot_df, x="loan_status", y="income_annum", ax=axes[0], inner="box")
axes[0].set_title("Годовой доход по статусу заявки")
axes[0].set_xlabel("Статус заявки")
axes[0].set_ylabel("Годовой доход, руб.")

sns.violinplot(data=plot_df, x="loan_status", y="loan_amount", ax=axes[1], inner="box")
axes[1].set_title("Сумма кредита по статусу заявки")
axes[1].set_xlabel("Статус заявки")
axes[1].set_ylabel("Сумма кредита, руб.")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "03_income_loan_violin.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
plot_df = df.copy()
plot_df["loan_status"] = plot_df["loan_status"].str.strip()

for status, color, label in [
    ("Approved", "steelblue", "Одобрен"),
    ("Rejected", "salmon", "Отказ"),
]:
    subset = plot_df[plot_df["loan_status"] == status]
    ax.scatter(subset["income_annum"], subset["loan_amount"],
               alpha=0.25, s=15, color=color, label=label)

ax.set_title("Доход заявителя vs Сумма кредита")
ax.set_xlabel("Годовой доход, руб.")
ax.set_ylabel("Сумма кредита, руб.")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_income_vs_loan_scatter.png", dpi=150)
plt.show()

In [ ]:
for col in ["education", "self_employed"]:
    table = pd.crosstab(df[col].str.strip(), df["loan_status"].str.strip())
    chi2, p, dof, _ = chi2_contingency(table)
    print(f"\n{col}:")
    print(table)
    print(f"Chi-square: {chi2:.2f}, p-value: {p:.2e}")

In [ ]:
cleaned = clean_loan_data(df)
featured = engineer_features(cleaned)

corr_cols = [
    "income_annum", "loan_amount", "loan_term",
    "loan_to_income", "payment_to_income", "total_assets_value",
    "avg_consumer_loan_rate_rub", "key_rate",
]
corr = featured[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
ax.set_title("Корреляционная матрица числовых признаков")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "05_correlation_heatmap.png", dpi=150)
plt.show()

**Промежуточный вывод:**

- **Гипотеза 1 подтверждена:** доход заявителя статистически значимо выше у одобренных заявок (Mann-Whitney U, p < 0.05). Одобренные заявки также характеризуются несколько меньшей запрашиваемой суммой относительно дохода.
- **Гипотеза 2 подтверждена:** хи-квадрат тест показал, что ни образование (p = 0.77), ни статус самозанятости (p = 1.00) **не связаны** с решением банка — оба признака статистически независимы от целевой переменной. Несмотря на это, они включены в модель как потенциально полезный контекст.

## 4. Очистка данных и feature engineering

Выполняем:
- удаление дубликатов и служебных полей;
- обработку выбросов методом IQR (в т.ч. отрицательных значений `residential_assets_value`);
- создание расчётных признаков: `loan_to_income`, `debt_burden_ratio`, `total_assets_value`, `payment_to_income`, `macro_spread`.

In [ ]:
new_features = [
    "loan_to_income", "debt_burden_ratio", "total_assets_value",
    "assets_per_income", "monthly_payment_estimate", "payment_to_income", "macro_spread",
]
featured[new_features].describe().T

> ⚠️ **Ограничение датасета:** в исходных данных отсутствует дата подачи заявки. Каждой записи назначается синтетический месяц детерминированно по `loan_id` (`loan_id % количество_месяцев`). Это означает, что макропоказатели ЦБ (`key_rate`, `avg_consumer_loan_rate_rub`, `macro_spread`) не отражают реальный экономический контекст конкретной заявки, а служат лишь дополнительными признаками общего характера.

## 5. Обучение моделей

Сравниваем три подхода:
- **Logistic Regression** — интерпретируемая базовая модель;
- **Random Forest** — ансамбль деревьев;
- **Gradient Boosting / XGBoost** — градиентный бустинг.

Без CIBIL score задача существенно сложнее: модель вынуждена опираться на финансовые соотношения и структуру активов заявителя.

In [ ]:
X_df, y = prepare_target(featured)
X = get_feature_matrix(X_df)

results, metrics_df, y_test, _ = train_and_evaluate(X, y)
metrics_df

In [ ]:
cv_scores = cross_validate_model(X, y, model_name=get_best_tree_model_name())
print(f"Кросс-валидация ({get_best_tree_model_name()}), ROC-AUC:")
print(cv_scores)
print(f"Среднее: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

In [ ]:
best_name = metrics_df.iloc[0]["model"]
best_result = results[best_name]
print(f"Лучшая модель: {best_name}")
print(best_result.report)

## 6. Визуализация результатов модели

In [ ]:
fig, ax = plt.subplots()
sns.barplot(data=metrics_df, x="model", y="roc_auc", ax=ax, hue="model", legend=False)
ax.set_title("Сравнение моделей по ROC-AUC")
ax.set_xlabel("Модель")
ax.set_ylabel("ROC-AUC")
ax.set_ylim(0.5, 1.0)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "06_model_comparison.png", dpi=150)
plt.show()

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(y_test, best_result.y_proba)

fig, ax = plt.subplots()
ax.plot(fpr, tpr, label=f"{best_name} (AUC={best_result.roc_auc:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Случайный классификатор")
ax.set_title("ROC-кривая лучшей модели")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "07_roc_curve.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots()
sns.heatmap(
    best_result.confusion,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Отказ", "Одобрен"],
    yticklabels=["Отказ", "Одобрен"],
    ax=ax,
)
ax.set_title(f"Матрица ошибок: {best_name}")
ax.set_xlabel("Предсказание")
ax.set_ylabel("Факт")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "08_confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
tree_name = get_best_tree_model_name()
tree_pipeline = results[tree_name].model
tree_model = tree_pipeline.named_steps["model"]

if hasattr(tree_model, "feature_importances_"):
    feature_names = tree_pipeline.named_steps["preprocessor"].get_feature_names_out()
    importances = pd.DataFrame({
        "feature": feature_names,
        "importance": tree_model.feature_importances_,
    }).sort_values("importance", ascending=False).head(10)

    fig, ax = plt.subplots()
    sns.barplot(data=importances, y="feature", x="importance",
                ax=ax, hue="feature", legend=False)
    ax.set_title(f"Топ-10 важных признаков ({tree_name})")
    ax.set_xlabel("Важность")
    ax.set_ylabel("Признак")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "09_feature_importance.png", dpi=150)
    plt.show()
    importances

## 7. Выводы

1. **Данные:** использован открытый синтетический датасет Kaggle (4 269 заявок), обогащённый макропоказателями ЦБ РФ через два API — SOAP (ключевая ставка) и REST (ставки по кредитам физлиц). CIBIL score исключён из анализа намеренно.

2. **EDA:** доход заявителя статистически значимо влияет на решение о выдаче кредита (Mann-Whitney U, p < 0.05). Образование и статус самозанятости не показали связи с целевой переменной (χ² тест: p = 0.77 и p = 1.00 соответственно) — эти признаки статистически независимы от решения банка.

3. **Feature engineering:** созданы признаки долговой нагрузки (`loan_to_income`, `debt_burden_ratio`), платёжеспособности (`payment_to_income`) и макро-спреда (`macro_spread`).

4. **Модель:** без CIBIL score качество предсказания ожидаемо ниже, чем при его наличии. Это реалистичная постановка задачи — реальные банковские данные без скорингового балла значительно сложнее для классификации.

5. **Ограничения:** датасет синтетический, правила одобрения заложены искусственно. Даты заявок назначены синтетически по `loan_id`, поэтому макропоказатели ЦБ отражают общий контекст периода, но не реальную связь с конкретными заявками.

6. **Практическая ценность:** модель демонстрирует, какие финансовые характеристики заявителя наиболее информативны при отсутствии кредитного скоринга.

### Использование AI

AI-инструменты использовались для проектирования структуры репозитория, генерации шаблонов модулей и подготовки Markdown-ячеек ноутбука. Логика анализа и интерпретация результатов выполнены командой проекта.